# Policy-Aware Function App Smoke Test

This notebook exercises the local Azure Functions app against the Enron email corpus with the repository's ODRL policy enforcement enabled.

Prerequisites:
- Ensure the Azure Functions runtime is installed (`func`)
- Populate `local.settings.json` with the Foundry and Cosmos values for your environment
- Start the app with `func start` before running this notebook
- Confirm that the Enron vector collection is available and the app is configured to use `all-MiniLM-L6-v2`
- Use a role and purpose that match the ODRL policy set in `odrl_policies/`

The current policy model includes:
- `business-observer` for metadata review / routing / triage
- `customer-support-specialist` for case handling / incident triage
- `privacy-compliance-analyst` for compliance, fraud, security, and privacy review
- `pii-data-governance-admin` for full access


In [1]:
import json
import requests

BASE_URL = "http://localhost:7071/api"

COMPLIANT_QUERY = "Summarize the key points from the California energy trading email thread."
DENIED_QUERY = "Export all personal information from the Enron corpus."
REDACTION_QUERY = "Provide the routing summary and the contact email for the primary deal lead."


In [2]:
health_response = requests.get(f"{BASE_URL}/health", timeout=30)
print("HEALTH STATUS:", health_response.status_code)
print(health_response.text)
assert health_response.status_code == 200


HEALTH STATUS: 200
{"status": "ok"}


In [3]:
compliant_payload = {
    "question": COMPLIANT_QUERY,
    "userRoles": ["privacy-compliance-analyst"],
    "purpose": "compliance_review",
    "action": "retrieve",
}

rag_response = requests.post(
    f"{BASE_URL}/rag",
    json=compliant_payload,
    timeout=120,
)

print("COMPLIANT RAG STATUS:", rag_response.status_code)
print(rag_response.text)
assert rag_response.status_code == 200
response_body = rag_response.json()
assert response_body["question"] == COMPLIANT_QUERY
assert "answer" in response_body
assert "sources" in response_body

print("ANSWER:")
print(response_body["answer"])
print("SOURCES:")
print(response_body["sources"])


COMPLIANT RAG STATUS: 200
{"question": "Summarize the key points from the California energy trading email thread.", "answer": "- CAISO announced that it had posted four key documents: its response to the CPUC/EOB Summer 2000 report, an action plan for demand response/generation/transmission projects, a May\u2013June 2000 report on California energy issues and performance, and testimony by CEO Terry Winter to the California Legislature. [enron_3518]\n- A Dow Jones report described California electricity demand as high amid tight supplies, with rolling brownouts and potential blackouts posing risks to industrial production, including computer-related industries. [enron_3790]\n- The report said about 26% of the Federal Reserve\u2019s industrial-production index was estimated using electricity-generation measures, meaning disruptions could affect both actual production and statistical measurements. [enron_3790]\n- California businesses had \u201cinterruptible rate plans,\u201d under which 

In [4]:
denied_payload = {
    "question": DENIED_QUERY,
    "userRoles": ["business-observer"],
    "purpose": "routing",
    "action": "export",
}

denied_response = requests.post(
    f"{BASE_URL}/rag",
    json=denied_payload,
    timeout=30,
)

print("DENIED RAG STATUS:", denied_response.status_code)
print(denied_response.text)
assert denied_response.status_code == 403
assert "policyDenied" in denied_response.json()


DENIED RAG STATUS: 403
{"error": "Policy denial: the request is not permitted by the active ODRL policy.", "policyDenied": true, "correlationId": "d1eb39a6-0f5b-4635-9b03-77053d05c5d1"}


In [5]:
redaction_payload = {
    "question": REDACTION_QUERY,
    "userRoles": ["business-observer"],
    "purpose": "routing",
    "action": "summarise",
}

redaction_response = requests.post(
    f"{BASE_URL}/rag",
    json=redaction_payload,
    timeout=120,
)

print("REDACTION RAG STATUS:", redaction_response.status_code)
print(redaction_response.text)
assert redaction_response.status_code == 200
body = redaction_response.json()
assert "answer" in body
assert "[REDACTED_EMAIL]" in body["answer"] or "[REDACTED_SSN]" in body["answer"] or "example.com" not in body["answer"].lower()
print("SANITIZED ANSWER:")
print(body["answer"])


REDACTION RAG STATUS: 200
{"question": "Provide the routing summary and the contact email for the primary deal lead.", "answer": "- **Routing summary:** Deal **#113186** should be rebooked and confirmed directly between **ENA and Direct Energy Marketing Limited**, removing the existing intermediary arrangement. **Grant Oh** can provide counterparty or contact information. [enron_5799]\n- **Primary deal lead contact:** The context does not identify a primary deal lead or provide that person\u2019s email address. The only explicit sender email in this thread is **[REDACTED_EMAIL]**. [enron_5799]", "sources": ["enron_9065", "enron_5229", "enron_4537", "enron_5799", "enron_4666", "enron_9413", "enron_278", "enron_2408"], "userRoles": ["business-observer"], "purpose": "routing", "correlationId": "3293588b-3e49-4ef5-8efd-b1efd26ffc6e"}
SANITIZED ANSWER:
- **Routing summary:** Deal **#113186** should be rebooked and confirmed directly between **ENA and Direct Energy Marketing Limited**, remov